# Tetrax OCR — Fine-Tuning on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TusharTemgire/Tetrax-OCR-itt/blob/main/Tetrax_OCR_Colab_Training.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Tetrax--OCR--itt-blue?logo=github)](https://github.com/TusharTemgire/Tetrax-OCR-itt)
[![Dataset: Sarvam Indic OCR](https://img.shields.io/badge/Dataset-sarvamai%2Findic--ocr--bench-orange)](https://huggingface.co/datasets/sarvamai/indic-ocr-bench)

Fine-tune high-performance OCR models on **Google Colab** (T4 / A100 / L4 GPU) using the [**`sarvamai/indic-ocr-bench`**](https://huggingface.co/datasets/sarvamai/indic-ocr-bench) English and Indian document dataset.

### Repository
**[https://github.com/TusharTemgire/Tetrax-OCR-itt.git](https://github.com/TusharTemgire/Tetrax-OCR-itt.git)**

---
### Workflow Summary:
1. **GPU Verification**: Verify CUDA acceleration on Colab's NVIDIA GPU.
2. **Mount Google Drive**: Save fine-tuned checkpoints permanently to Drive.
3. **Repository Setup**: Clone `Tetrax-OCR-itt` and install dependencies.
4. **Dataset Preparation**: Extract and format `sarvamai/indic-ocr-bench` for Indian languages and English.
5. **Option A (PaddleOCR)**: Train SVTR-LCNet / PP-OCRv4 recognition models and export static inference weights.
6. **Option B (TrOCR)**: In-memory fine-tuning with Hugging Face Vision Encoder-Decoder Transformers.
7. **Evaluation**: Benchmark Character Error Rate (CER), Word Error Rate (WER), and accuracy.
8. **Inference Test**: Run real-time recognition on sample test documents.
9. **Export & Download**: Export static inference models and download zip archives.

## Step 1: Verify Colab GPU Acceleration
Select **Runtime > Change runtime type > T4 GPU** (or A100 / L4).

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive
Mounting Google Drive ensures your fine-tuned model checkpoints (`.pdparams` / Hugging Face weights) are saved to your Drive and will not be lost if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create permanent directories in your Google Drive for checkpoints and exported models
!mkdir -p /content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints
!mkdir -p /content/drive/MyDrive/tetrax_ocr_models/paddle_inference
!mkdir -p /content/drive/MyDrive/tetrax_ocr_models/trocr_checkpoint

print("Google Drive mounted and checkpoint directories created successfully!")

## Step 3: Clone the Tetrax OCR Repository
Clones [`https://github.com/TusharTemgire/Tetrax-OCR-itt.git`](https://github.com/TusharTemgire/Tetrax-OCR-itt.git) into `/content/tetrax-ocr`.

In [ ]:
!rm -rf /content/tetrax-ocr
!git clone https://github.com/TusharTemgire/Tetrax-OCR-itt.git /content/tetrax-ocr
%cd /content/tetrax-ocr

## Step 4: Install Dependencies & Setup Environment
Installs PyTorch, Hugging Face Transformers, PaddlePaddle GPU (CUDA 12 build for Colab), PaddleOCR, and computer vision libraries.

In [ ]:
# Upgrade pip
!python -m pip install --upgrade pip

# Install project requirements
%cd /content/tetrax-ocr
!pip install -r requirements.txt

# Install PaddlePaddle GPU for Colab (CUDA 12.x) with automatic fallback
!pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ ||  !pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu120/ ||  !pip install paddlepaddle-gpu ||  !pip install paddlepaddle

!pip install "paddleocr>=2.7" 

### Verify CUDA Acceleration in Both Engines

In [ ]:
import torch

print("=" * 50)
print("PYTORCH ACCELERATION:")
print("  CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  Device Name:    ", torch.cuda.get_device_name(0))
    print("  Device Count:   ", torch.cuda.device_count())

print("\nPADDLEPADDLE ACCELERATION:")
try:
    import paddle
    print("  Paddle Version: ", paddle.__version__)
    print("  CUDA Compiled:  ", paddle.is_compiled_with_cuda())
    if paddle.is_compiled_with_cuda():
        print("  Device Name:    ", paddle.device.cuda.get_device_name(0))
        print("  Device Count:   ", paddle.device.cuda.device_count())
    else:
        print("  [!] Running in CPU mode. Make sure a GPU is selected in Runtime > Change runtime type.")
except ModuleNotFoundError:
    print("  [!] ModuleNotFoundError: 'paddle' is not installed in the active Colab environment.")
    print("  Please run 'Step 4: Install Dependencies' above, or run:")
    print("    !pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/")
print("=" * 50)

## Step 5: Download & Prepare Complete `sarvamai/indic-ocr-bench` Dataset
The [**`sarvamai/indic-ocr-bench`**](https://huggingface.co/datasets/sarvamai/indic-ocr-bench) dataset includes all 22 official Indian languages plus English:
* **`split="test"`**: The complete benchmark containing **6,909 document images (~638 MB)** across all languages.
* **`languages="all"`**: Extracts all 22 Indian scripts and builds the complete Unicode character dictionary (`sarvam_dict.txt`).

*(Tip: If you ever want a fast 2-minute trial run, replace `--split test --languages all` with `--split small_representative --languages hindi,english`).*

In [ ]:
%cd /content/tetrax-ocr

# Extract and prepare the COMPLETE dataset (all 6,909 images across all 22 Indian languages + English)
!python -m training.load_sarvam_dataset \
  --split test \
  --languages all \
  --output-dir datasets/sarvam_indic

### Verify Dataset & Character Dictionary
Inspect the extracted files, train/val/test splits, and character dictionary:

In [ ]:
!ls -lh datasets/sarvam_indic/paddle_rec/
print("\n--- Total character dictionary size: ---")
!wc -l datasets/sarvam_indic/dicts/sarvam_dict.txt

print("\n--- First 5 training labels: ---")
!head -n 5 datasets/sarvam_indic/paddle_rec/train.txt

print("\n--- First 20 dictionary entries: ---")
!head -n 20 datasets/sarvam_indic/dicts/sarvam_dict.txt

## Step 6: Option A — Fine-Tune PaddleOCR on Complete Indic Dataset
PaddleOCR is the recommended high-throughput engine for production documents across Indian scripts.

### 6.1 Clone PaddleOCR Training Framework & Download Pretrained Base Weights

In [ ]:
!rm -rf /content/PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
%cd /content/PaddleOCR
!pip install -r requirements.txt

# Download pretrained base multilingual PP-OCR training weights
!mkdir -p pretrain_models
%cd pretrain_models
!wget -nc https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/multilingual_PP-OCRv4_rec_train.tar
!tar -xf multilingual_PP-OCRv4_rec_train.tar
%cd /content/PaddleOCR

### 6.2 Launch PaddleOCR Fine-Tuning
Checkpoints are saved directly to your mounted Google Drive (`/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints`).

In [ ]:
%cd /content/PaddleOCR

!python tools/train.py \
  -c /content/tetrax-ocr/configs/rec_sarvam_indic.yml \
  -o Global.use_gpu=True \
     Global.epoch_num=30 \
     Global.pretrained_model=./pretrain_models/multilingual_PP-OCRv4_rec_train/best_accuracy \
     Global.character_dict_path=/content/tetrax-ocr/datasets/sarvam_indic/dicts/sarvam_dict.txt \
     Global.save_model_dir=/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints \
     Train.dataset.data_dir=/content/tetrax-ocr \
     Train.dataset.label_file_list='["/content/tetrax-ocr/datasets/sarvam_indic/paddle_rec/train.txt"]' \
     Train.dataset.transforms.1.MultiLabelEncode.gt_label_path=/content/tetrax-ocr/datasets/sarvam_indic/dicts/sarvam_dict.txt \
     Eval.dataset.data_dir=/content/tetrax-ocr \
     Eval.dataset.label_file_list='["/content/tetrax-ocr/datasets/sarvam_indic/paddle_rec/val.txt"]' \
     Eval.dataset.transforms.1.MultiLabelEncode.gt_label_path=/content/tetrax-ocr/datasets/sarvam_indic/dicts/sarvam_dict.txt

### 6.3 Export Inference Model for Production Deployment
Converts dynamic training checkpoints to static inference files (`inference.pdmodel`, `inference.pdiparams`).

In [ ]:
%cd /content/PaddleOCR

!python tools/export_model.py \
  -c /content/tetrax-ocr/configs/rec_sarvam_indic.yml \
  -o Global.checkpoints=/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints/best_accuracy \
     Global.save_inference_dir=/content/drive/MyDrive/tetrax_ocr_models/paddle_inference

print("Export completed! Model files saved to /content/drive/MyDrive/tetrax_ocr_models/paddle_inference")
!ls -lh /content/drive/MyDrive/tetrax_ocr_models/paddle_inference

## Step 7: Option B — In-Memory Fine-Tuning with TrOCR (Hugging Face)
Fine-tunes a Vision Transformer (VisionEncoderDecoderModel) directly using the complete `sarvamai/indic-ocr-bench` dataset in memory without local image files.

In [ ]:
%cd /content/tetrax-ocr

!python -m training.train_trocr \
  --dataset "sarvamai/indic-ocr-bench" \
  --split test \
  --languages all \
  --output-dir /content/drive/MyDrive/tetrax_ocr_models/trocr_checkpoint \
  --epochs 3 \
  --batch-size 8 \
  --lr 5e-5

## Step 8: Benchmark Fine-Tuned Model Accuracy (CER, WER, Exact Match)
Benchmark accuracy across all Indian languages and English on the full Sarvam benchmark using your fine-tuned model:

In [ ]:
%cd /content/tetrax-ocr

# Benchmark the fine-tuned PaddleOCR model across all languages on the full test split
!python -m training.benchmark_sarvam \
  --split test \
  --languages all \
  --max-samples 100 \
  --rec-model-dir /content/drive/MyDrive/tetrax_ocr_models/paddle_inference \
  --rec-char-dict-path datasets/sarvam_indic/dicts/sarvam_dict.txt

## Step 9: Test Inference with Fine-Tuned Model on Test Document
Runs real-time inference on a document sample from the full benchmark dataset using your fine-tuned model weights:

In [ ]:
%cd /content/tetrax-ocr

import cv2
import numpy as np
from datasets import load_dataset
from app.ocr_engine import extract_image

# Load test sample from the full Sarvam Indic OCR dataset
ds = load_dataset("sarvamai/indic-ocr-bench", split="test")
sample = ds[0]
pil_img = sample["image"]
ground_truth = sample.get("gt", "")
lang = sample.get("language", "en")

print(f"Document Language: {lang}")
print(f"Ground Truth:\n{ground_truth}\n")

# Run inference with YOUR fine-tuned exported PaddleOCR model
img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)

finetuned_dir = "/content/drive/MyDrive/tetrax_ocr_models/paddle_inference"
dict_path = "/content/tetrax-ocr/datasets/sarvam_indic/dicts/sarvam_dict.txt"

result = extract_image(
    img_bgr,
    lang="hi" if "hi" in lang.lower() else "en",
    apply_preprocessing=True,
    rec_model_dir=finetuned_dir,
    rec_char_dict_path=dict_path,
)

print("=" * 50)
print("FINE-TUNED PADDLEOCR PREDICTION:")
print(result["text"])
print("=" * 50)
print(f"Detected {result.get('line_count', len(result.get('lines', [])))} text line(s).")

### Optional: Test Fine-Tuned TrOCR Transformer (Option B)

In [ ]:
# Test inference with fine-tuned TrOCR checkpoint
import os
from models.test_fine_tuned_model import extract_text_with_finetuned_model

trocr_dir = "/content/drive/MyDrive/tetrax_ocr_models/trocr_checkpoint"
if os.path.exists(trocr_dir):
    test_img_path = "/content/sample_test.png"
    pil_img.save(test_img_path)
    trocr_pred = extract_text_with_finetuned_model(test_img_path, trocr_dir)
    print("=" * 50)
    print("FINE-TUNED TrOCR PREDICTION:")
    print(trocr_pred)
    print("=" * 50)
else:
    print(f"TrOCR checkpoint not found at {trocr_dir} (run Step 7 if you want to test TrOCR)")

## Step 10: Download Trained Models Archive
Archive your fine-tuned models to zip for easy download or deployment:

In [ ]:
!zip -r /content/tetrax_ocr_inference.zip /content/drive/MyDrive/tetrax_ocr_models/paddle_inference

from google.colab import files
# Uncomment to trigger direct browser download:
# files.download('/content/tetrax_ocr_inference.zip')